### 07. Exploring the Malavasi et al. 2020 filament catalog 

This notebook explores the use of the filament catalog generated using DisPerSe.

**Author**: Soumya Shreeram <br/>
**Date created**: 15th July 2022 <br/>
**Last edited**: tbu <br/>

In [101]:
# astropy modules
import astropy.units as u
import astropy.io.fits as fits
from astropy.table import Table, Column, QTable

import numpy as np

# system imports
import os
import sys
import importlib as ib
import glob

In [3]:
import filaments as fo

In [4]:
%load_ext autoreload
%autoreload 2

In [51]:
base_dir = "/data53s/shreeram/Filament_stacking"
keyword = "cp"

The filaments are detected for two samples: the SDSS DR7 (Legacy Main Galaxy Sample; `legacy_north_dis`) and the SDSS DR12 (LOWZ+CMASS; `lc_north_dis`)

The following terminologies are used in the notebook:

1. Critical points: the points at which the gradient of the galaxy density field vanishes. There are three types of critical points: maxima, minima, type 1 and type 2 saddles (local density minima bound to walls and filaments respectively).
2. Bifurcation points: points where filaments intersected in post-processing.
3. Persistance ratio = signal-to-noise threshold: defines the level to eliminate spurious filaments (depending on how close they are to the noise persistance distrubtion). $\sim 5\%$ at 2$\sigma$, 0.006$\%$ at 4$\sigma$.

Filaments are the connections between the following critical points: maxima and type 2 saddles.

In [82]:
this_file = fo.DisPerSEcatalog()

INFO:DisPerSEcatalog:<DisPerSEcatalog(data_set=legacy_north_dis, smoothing_density_f=SD1, persistence=3, smoothing_skeleton=S001)>


Reading the ascii file in it's orginal format.

In [7]:
output_ncrit_list, list_entry_lengths = [], []
f = open(this_file.catalog_filename, 'r')
for l, line in enumerate(f):
    line = line.strip()
    if (l > this_file.ncrit_line_no) and l < (this_file.nfil_line_no-1):
        line = np.array(line.split()).astype(float)
        list_entry_lengths.append(len(line))
        output_ncrit_list.append(line)
f.close()
output_ncrit_list = np.array(output_ncrit_list, dtype=object)

Using an index to mark each code block within the ascii file.

In [8]:
set_group = []
counter = 0
for idx, val in enumerate(list_entry_lengths):
    if val == 7:
        counter += 1
    set_group.append(counter)

Converting each variable stored in the ascii file into an array, which is then fed into the object `Table` to make a fits file.

In [35]:
grouped_idx_arr = np.array([], dtype=object)

cp_type_arr = np.array([])
pos_x_arr = np.array([])
pos_y_arr = np.array([])
pos_z_arr = np.array([])
nfil_cp_arr = np.array([])

cp_idx_end_2Darr, fil_idx_2Darr = [], []

for i in range(3):
    # indicies belonging to the same block
    grouped_idx = np.where(np.array(set_group) == (i + 1))[0]
    
    # variables in that specific block
    cp_type_arr = np.append(cp_type_arr, int(output_ncrit_list[grouped_idx[0]][0]))
    pos_x_arr = np.append(pos_x_arr, output_ncrit_list[grouped_idx[0]][1])
    pos_y_arr = np.append(pos_y_arr, output_ncrit_list[grouped_idx[0]][2])
    pos_z_arr = np.append(pos_z_arr, output_ncrit_list[grouped_idx[0]][3])
    
    # the number of filaments connected to the critical point 
    nfil_cp_arr = np.append(nfil_cp_arr, int(output_ncrit_list[grouped_idx[1]][0]))
    cp_idx_end_arr, fil_idx_arr = [], []
    for n in range(nfil_cp):
        number = int(2+n)
        
        cp_idx_end, fil_idx = output_ncrit_list[grouped_idx[number]]
        cp_idx_end_arr.append(int(cp_idx_end)) 
        fil_idx_arr.append(int(fil_idx))
    
    cp_idx_end_2Darr.append(cp_idx_end_arr)
    fil_idx_2Darr.append(fil_idx_arr)

Here is the dummy table that is formed and then saved into a fits file

In [75]:
dummy = QTable(data=[cp_type_arr,
            pos_x_arr,
            pos_y_arr,
            pos_z_arr,
            nfil_cp_arr,
           cp_idx_end_2Darr
           ], 
      names = [ 'cp_type', 'pos_x', 'pos_y', 'pos_z', 'nfil_cp', 'cp_idx_end_2Darr'],
      )

In [76]:
dummy

cp_type,pos_x,pos_y,pos_z,nfil_cp,cp_idx_end_2Darr [2]
float64,float64,float64,float64,float64,int64
3.0,493.123,202.202,-112.7,2.0,742 .. 769
3.0,251.713,269.898,-172.19,4.0,1272 .. 2985
3.0,582.303,-219.582,518.861,2.0,3860 .. 6203


The above cells have been implemented into the class DisperseCatalog. Refer to the library for further details and the example of it's implementation is shown in the file "reformatting_Malavasi_catalog2fit.py"

In [77]:
filename = f"{this_file.data_set}_{this_file.smoothing_density_f}_{this_file.persistence}_{this_file.smoothing_skeleton}_{keyword}" 
dummy.write(f'{this_file.catalog_dir_reformatted}/{filename}.fits', format='fits', overwrite=True)

Cell to see how to change the filament section of the ascii file

In [88]:
f = open(this_file.catalog_filename, 'r')
output_nfil_list, set_grp2 = [], []
counter = 0
for l, line in enumerate(f):
    if (l > this_file.nfil_line_no) and l < (this_file.nf_line_no-1):
        line = line.strip()
        line = np.array(line.split()).astype(float)
        list_entry_lengths2.append(len(line))
        output_nfil_list.append(line)
f.close()
output_nfil_list = np.array(output_nfil_list, dtype=object)